<a href="https://colab.research.google.com/github/LepingWan/testrepo/blob/main/Panini_Course_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



You will work with both supplied 100-question packages. Read
`PROJECT_HANDOUT.pdf` before editing this notebook. Every section
below corresponds to a numbered handout question, and every
explanation cell is part of the submission.

## How to run this notebook on a free Colab GPU

The neural stages are deliberately separated. Never keep two Qwen
models resident at once.

```text
CPU audit and indices
        ↓
Stage A: decomposer → save JSONL → unload model and clear CUDA
        ↓
Stage B: reranker   → save traces/rankings → unload and clear CUDA
        ↓
Stage C: answerer   → save predictions → unload and clear CUDA
```

During development, keep `QUESTION_LIMIT = 2`. Once your tests and
output schemas are correct, set it to `None`, enable exactly one
neural stage, and run all cells. Completed question IDs are read
from JSONL checkpoints, so reconnecting and running again resumes
rather than starts over. The supplied query vectors mean that you
must not load or regenerate the Qwen embedding model.

In [ ]:
# Run controls. Change one neural stage at a time.
# Defaults are cheap and restart-safe: they let Runtime > Run all complete
# without loading Qwen models. Turn on one expensive stage at a time in Colab.
RUN_DECOMPOSITION_STAGE = False
RUN_RERANK_AND_RICR_STAGE = False
RUN_ANSWER_STAGE = False
RUN_ABLATIONS = False

# Use 2 while developing. Set to None for every required final run.
QUESTION_LIMIT = 2
MOUNT_DRIVE_IN_COLAB = False
STRICT_SMOKE_TESTS = False
DATASETS = ('2wiki', 'musique')

# Frozen default configuration from the project specification.
BEAM_WIDTH = 5
CANDIDATES_PER_HOP = 15
RETRIEVAL_POOL = 60
RRF_CONSTANT = 60.0
MULTI_PARENT_THRESHOLD = 0.3


In [ ]:
from pathlib import Path
import gc, json, os, platform, shutil, subprocess, sys, time

IN_COLAB = 'google.colab' in sys.modules
DEFAULT_REPO_URL = 'https://github.com/YigitTurali/panini-course-project.git'
PANINI_REPO_URL = os.environ.get('PANINI_REPO_URL', DEFAULT_REPO_URL)

if IN_COLAB:
    REPO_ROOT = Path(os.environ.get('PANINI_REPO_ROOT', '/content/panini-course-project'))
    if not (REPO_ROOT / 'manifest.json').exists():
        if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
            raise FileNotFoundError(
                f'{REPO_ROOT} exists but does not look like the Panini package. '
                'Set PANINI_REPO_ROOT to a clean path or delete the partial clone.'
            )
        subprocess.run([
            'git', 'clone', '--depth', '1',
            PANINI_REPO_URL,
            str(REPO_ROOT),
        ], check=True)

    requirements = REPO_ROOT / 'requirements-colab.txt'
    if requirements.exists():
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '-q', '-r',
            str(requirements),
        ], check=True)

    if MOUNT_DRIVE_IN_COLAB:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=True)
            WORK_ROOT = Path('/content/drive/MyDrive/panini-course-project-work')
        except Exception as error:
            print(f'Drive mount failed ({error!r}); falling back to temporary /content storage.')
            WORK_ROOT = Path('/content/panini-course-project-work')
    else:
        WORK_ROOT = Path('/content/panini-course-project-work')
else:
    here = Path.cwd().resolve()
    if (here / 'manifest.json').exists():
        REPO_ROOT = here
    else:
        source = here
        while source != source.parent and not (source / 'course_project').exists():
            source = source.parent
        if not (source / 'course_project').exists():
            raise FileNotFoundError(
                'Run from the public Panini repository, the gsw-memory checkout, '
                'or Colab. This lightweight testrepo contains only the notebook.'
            )
        REPO_ROOT = source / 'course_project/release/panini_2wiki_100'
    WORK_ROOT = Path(os.environ.get('PANINI_STUDENT_WORK', here / 'panini-student-work'))

PACKAGE_ROOTS = {
    '2wiki': REPO_ROOT,
    'musique': REPO_ROOT / 'packages/panini_musique_100'
        if (REPO_ROOT / 'packages').exists()
        else REPO_ROOT.parent / 'panini_musique_100',
}
sys.path.insert(0, str(REPO_ROOT))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Keep graded source edits in persistent work storage, not only in the
# disposable /content clone. If this notebook is run from a team checkout
# containing student_code/ricr.py, that version seeds the persistent copy.
STUDENT_CODE_ROOT = WORK_ROOT / 'student_code'
VERSIONED_STUDENT_CODE_ROOT = Path.cwd().resolve() / 'student_code'
STUDENT_CODE_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_RICR = STUDENT_CODE_ROOT / 'ricr.py'
RUNTIME_RICR = REPO_ROOT / 'panini_course' / 'ricr.py'
VERSIONED_RICR = VERSIONED_STUDENT_CODE_ROOT / 'ricr.py'
if VERSIONED_RICR.exists():
    shutil.copy2(VERSIONED_RICR, PERSISTENT_RICR)
elif not PERSISTENT_RICR.exists() and RUNTIME_RICR.exists():
    shutil.copy2(RUNTIME_RICR, PERSISTENT_RICR)
if PERSISTENT_RICR.exists() and RUNTIME_RICR.exists():
    shutil.copy2(PERSISTENT_RICR, RUNTIME_RICR)

PERSISTENT_TESTS = STUDENT_CODE_ROOT / 'test_student_ricr.py'
VERSIONED_TESTS = VERSIONED_STUDENT_CODE_ROOT / 'test_student_ricr.py'
if VERSIONED_TESTS.exists():
    shutil.copy2(VERSIONED_TESTS, PERSISTENT_TESTS)
elif not PERSISTENT_TESTS.exists():
    PERSISTENT_TESTS.write_text(
        "import pytest\n\n"
        "@pytest.mark.skip(reason='Replace with your own reconciliation/RICR test')\n"
        "def test_student_failure_case():\n"
        "    pass\n",
        encoding='utf-8',
    )
RUNTIME_STUDENT_TESTS = REPO_ROOT / 'tests' / 'test_student_work.py'
if PERSISTENT_TESTS.exists() and RUNTIME_STUDENT_TESTS.parent.exists():
    shutil.copy2(PERSISTENT_TESTS, RUNTIME_STUDENT_TESTS)
print({
    'repo': str(REPO_ROOT), 'work': str(WORK_ROOT), 'colab': IN_COLAB,
    'repo_url': PANINI_REPO_URL,
    'edit_ricr_here': str(PERSISTENT_RICR),
    'edit_tests_here': str(PERSISTENT_TESTS),
})


## Shared checkpoint and memory helpers

These helpers are infrastructure, not answers to a graded
algorithm. Each expensive stage appends one complete record at a
time. Do not keep results only in Python variables.

In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def append_jsonl(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')

def completed_ids(path, *, configuration=None):
    rows = read_jsonl(path)
    if configuration is not None:
        rows = [row for row in rows if row.get('configuration') == configuration]
    return {str(row['question_id']) for row in rows}

def release_gpu(*objects):
    # Delete caller-owned model variables before calling this helper.
    for value in objects:
        del value
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

def selected_questions(package):
    rows = package.questions('public') + package.questions('held_out')
    return rows if QUESTION_LIMIT is None else rows[:QUESTION_LIMIT]

def require_full_run():
    assert QUESTION_LIMIT is None, 'Set QUESTION_LIMIT = None before producing final files.'

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display
from panini_course import CoursePackage

packages = {name: CoursePackage(path) for name, path in PACKAGE_ROOTS.items()}
for name, package in packages.items():
    print(name, package.manifest['counts'])

## Question 1 — understand and verify the two packages (6 points)

Audit stable identifiers and artifact alignment before doing any
retrieval. Equal row counts alone are not sufficient. Inspect one
public question, held-out question, entity, and QA record from each
dataset. Confirm that held-out rows expose no answer or gold
evidence fields.

In [ ]:
def audit_package(package, dataset):
    # TODO Q1:
    # 1. Check uniqueness of entity_uid and qa_uid.
    # 2. Check exact ID-set equality between metadata and every
    #    embedding/index ID file used later.
    # 3. Check matrix row counts and embedding dimensions.
    # 4. Check held-out field names for label leakage.
    # 5. Return one flat dictionary for a two-row audit table.
    entities = package.entities()
    qa_pairs = package.qa_pairs()
    entity_ids = [row['entity_uid'] for row in entities]
    qa_ids = [row['qa_uid'] for row in qa_pairs]

    root = package.root
    embed_entity_ids = json.loads((root / 'embeddings' / 'entity_ids.json').read_text())
    embed_qa_ids = json.loads((root / 'embeddings' / 'qa_ids.json').read_text())
    index_entity_ids = json.loads((root / 'indices' / 'entity_ids.json').read_text())
    index_qa_ids = json.loads((root / 'indices' / 'qa_ids.json').read_text())

    entity_matrix = np.load(root / 'embeddings' / 'entity_embeddings.npy')
    qa_matrix = np.load(root / 'embeddings' / 'qa_embeddings.npy')
    embed_manifest = json.loads((root / 'embeddings' / 'embedding_manifest.json').read_text())

    counts = package.manifest['counts']
    held_out_rows = package.questions('held_out')
    leakage_fields = {'answer', 'answer_aliases', 'supporting_facts', 'supporting_document_ids', 'evidences'}
    leaked_fields = sorted({field for row in held_out_rows for field in row if field in leakage_fields})

    return {
        'dataset': dataset,
        'entity_uid_unique': len(set(entity_ids)) == len(entity_ids),
        'qa_uid_unique': len(set(qa_ids)) == len(qa_ids),
        'entity_ids_match_embeddings': set(entity_ids) == set(embed_entity_ids),
        'entity_ids_match_indices': set(entity_ids) == set(index_entity_ids),
        'qa_ids_match_embeddings': set(qa_ids) == set(embed_qa_ids),
        'qa_ids_match_indices': set(qa_ids) == set(index_qa_ids),
        'entity_rows_match_counts': entity_matrix.shape[0] == len(entity_ids) == counts['entities'],
        'qa_rows_match_counts': qa_matrix.shape[0] == len(qa_ids) == counts['qa_pairs'],
        'embedding_dimension': int(entity_matrix.shape[1]),
        'embedding_dims_consistent': entity_matrix.shape[1] == qa_matrix.shape[1] == embed_manifest['dimension'],
        'questions_public_count_matches': len(package.questions('public')) == counts['questions_public'],
        'questions_held_out_count_matches': len(held_out_rows) == counts['questions_held_out'],
        'held_out_leakage_fields': leaked_fields,
        'held_out_has_no_leakage': len(leaked_fields) == 0,
    }

# Uncomment after implementing audit_package.
audit_table = pd.DataFrame([
    audit_package(package, dataset)
    for dataset, package in packages.items()
])
display(audit_table)

### Your written response for Question 1

Replace this cell with a **150–200-word explanation in your own words**.
Explain why stable-ID set equality catches failures that shape checks miss, and describe one plausible silent alignment bug.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Question 2 — build the GSW network and reconcile entities (12 points)

Keep three graph objects separate: the document-local occurrence
projection, the supplied exact-surface sensitivity baseline, and
your conservative reconciliation. Reconciliation is an analysis
mapping only; it must never rewrite the packaged IDs.

In [ ]:
from panini_course.graph import (
    build_entity_projection,
    build_native_gsw_graph,
    build_unreconciled_entity_projection,
)

graph_sets = {}
for dataset, package in packages.items():
    native = build_native_gsw_graph(package.gsw_paths())
    unreconciled = build_unreconciled_entity_projection(native)
    exact_surface = build_entity_projection(native)
    graph_sets[dataset] = {
        'native': native,
        'unreconciled': unreconciled,
        'exact_surface': exact_surface,
    }
    print(dataset, {name: (g.number_of_nodes(), g.number_of_edges())
                    for name, g in graph_sets[dataset].items()})

def conservative_entity_mapping(native_graph):
    # TODO Q2: return (occurrence_uid_to_global_id, decision_rows).
    # Your rule must use surface, node type/role compatibility, and
    # local-neighborhood evidence. Block generic attribute values.
    import re
    from panini_course.graph import canonical_entity_name

    generic_role_markers = {
        'date', 'time', 'number', 'quantity', 'year',
        'range', 'count', 'duration', 'percentage', 'age',
    }

    def role_categories(attrs):
        return {str(role.get('role', '')).casefold().strip() for role in attrs.get('roles', [])}

    def state_tokens(attrs):
        tokens = set()
        for role in attrs.get('roles', []):
            for state in role.get('states', []):
                tokens.update(re.findall(r'\w+', str(state).casefold()))
        return tokens

    def is_generic(attrs, categories):
        if any(any(marker in category for marker in generic_role_markers) for category in categories):
            return True
        return len(canonical_entity_name(str(attrs.get('name', '')))) <= 2

    groups = {}
    for node, attrs in native_graph.nodes(data=True):
        if attrs.get('node_type') == 'verb_phrase':
            continue
        canonical = canonical_entity_name(str(attrs.get('name', node)))
        if canonical:
            groups.setdefault(canonical, []).append(node)

    occurrence_uid_to_global_id = {}
    decision_rows = []
    for canonical, nodes in groups.items():
        if len(nodes) < 2:
            occurrence_uid_to_global_id[nodes[0]] = nodes[0]
            continue
        clusters = []
        for node in nodes:
            attrs = native_graph.nodes[node]
            categories = role_categories(attrs)
            tokens = state_tokens(attrs)
            generic = is_generic(attrs, categories)
            placed = False
            if not generic:
                for cluster in clusters:
                    rep_attrs, rep_categories, rep_tokens = cluster[1], cluster[2], cluster[3]
                    if (
                        attrs.get('node_type') == rep_attrs.get('node_type')
                        and categories & rep_categories
                        and tokens & rep_tokens
                    ):
                        cluster[0].append(node)
                        placed = True
                        break
            if not placed:
                clusters.append([[node], attrs, categories, tokens])

        for index, (members, _, _, _) in enumerate(clusters):
            documents = sorted({native_graph.nodes[n].get('document_id') for n in members})
            merged = len(members) > 1 and len(documents) > 1
            global_id = f'{canonical}::{index}' if len(clusters) > 1 else canonical
            for node in members:
                occurrence_uid_to_global_id[node] = global_id
            decision_rows.append({
                'canonical_name': canonical,
                'global_id': global_id,
                'occurrences': members,
                'documents': documents,
                'merged': merged,
            })

    return occurrence_uid_to_global_id, decision_rows

def aggregate_projection(unreconciled_graph, mapping):
    # TODO Q2: build a weighted graph after applying the mapping.
    # Preserve occurrence counts and contributing document IDs.
    import networkx as nx

    aggregated = nx.Graph()
    for node, attrs in unreconciled_graph.nodes(data=True):
        global_id = mapping.get(node, node)
        if global_id not in aggregated:
            aggregated.add_node(
                global_id,
                name=attrs.get('name'),
                node_type=attrs.get('node_type'),
                occurrences=0,
                documents=set(),
            )
        aggregated.nodes[global_id]['occurrences'] += 1
        aggregated.nodes[global_id]['documents'].add(attrs.get('document_id'))

    for left, right, attrs in unreconciled_graph.edges(data=True):
        left_id = mapping.get(left, left)
        right_id = mapping.get(right, right)
        if left_id == right_id:
            continue
        weight = attrs.get('weight', 1)
        if aggregated.has_edge(left_id, right_id):
            aggregated[left_id][right_id]['weight'] += weight
        else:
            aggregated.add_edge(left_id, right_id, weight=weight)

    return aggregated

# TODO Q2: run the fixed-seed manual audit of at least 15 proposed
# cross-document merges and add the conservative graphs to graph_sets.
import random

reconciliation_audits = {}
for dataset in packages:
    mapping, decisions = conservative_entity_mapping(graph_sets[dataset]['native'])
    graph_sets[dataset]['conservative'] = aggregate_projection(
        graph_sets[dataset]['unreconciled'], mapping)

    merges = [row for row in decisions if row['merged']]
    rng = random.Random(232)
    sample_size = min(15, len(merges))
    reconciliation_audits[dataset] = pd.DataFrame(rng.sample(merges, sample_size))
    print(dataset, {name: (g.number_of_nodes(), g.number_of_edges())
                    for name, g in graph_sets[dataset].items()})
    display(reconciliation_audits[dataset])

### Your written response for Question 2

Replace this cell with a **200–300-word explanation in your own words**.
State your decision rule precisely. Use one false merge and one missed alias to explain why reconciliation changes the mathematical graph rather than merely cleaning labels.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Question 3 — analyze the structured-memory network (12 points)

Perform the full sensitivity analysis on 2Wiki. For MuSiQue,
produce the compact transfer table required by the handout. A
centrality ranking is not self-interpreting: inspect the semantic
role, contributing documents, and reconciliation decision for each
apparent hub.

In [ ]:
def network_statistics(graph):
    # TODO Q3: components, giant component, isolates, component-size
    # summary, degree summary, clustering, and assortativity.
    import statistics

    if graph.is_directed():
        components = list(nx.weakly_connected_components(graph))
    else:
        components = list(nx.connected_components(graph))

    sizes = sorted(len(component) for component in components)
    n_nodes = graph.number_of_nodes()
    giant_size = sizes[-1] if sizes else 0
    simple = nx.Graph(graph)
    degrees = [degree for _, degree in simple.degree()]

    return {
        'nodes': n_nodes,
        'edges': graph.number_of_edges(),
        'components': len(components),
        'giant_component_size': giant_size,
        'giant_component_fraction': giant_size / n_nodes if n_nodes else 0.0,
        'isolated_nodes': sum(1 for size in sizes if size == 1),
        'component_size_min': sizes[0] if sizes else 0,
        'component_size_median': statistics.median(sizes) if sizes else 0,
        'component_size_mean': statistics.fmean(sizes) if sizes else 0.0,
        'component_size_max': giant_size,
        'degree_min': min(degrees) if degrees else 0,
        'degree_mean': statistics.fmean(degrees) if degrees else 0.0,
        'degree_median': statistics.median(degrees) if degrees else 0,
        'degree_max': max(degrees) if degrees else 0,
        'average_clustering': nx.average_clustering(simple) if simple.number_of_nodes() else 0.0,
        'degree_assortativity': (
            nx.degree_assortativity_coefficient(simple)
            if simple.number_of_edges() else float('nan')
        ),
    }

def centrality_audit(graph, top_n=10, seed=232):
    # TODO Q3: weighted degree, PageRank, and exact or fixed-seed
    # approximate betweenness, followed by semantic hub labels.
    simple = nx.Graph(graph)
    weighted_degree = dict(simple.degree(weight='weight'))
    pagerank = nx.pagerank(simple, weight='weight')
    if simple.number_of_nodes() > 500:
        betweenness = nx.betweenness_centrality(
            simple, k=min(200, simple.number_of_nodes()), weight='weight', seed=seed)
        betweenness_exact = False
    else:
        betweenness = nx.betweenness_centrality(simple, weight='weight')
        betweenness_exact = True

    def top(mapping):
        ranked = sorted(mapping.items(), key=lambda item: item[1], reverse=True)[:top_n]
        return [
            {
                'node': node,
                'score': score,
                'name': simple.nodes[node].get('name', node),
                'node_type': simple.nodes[node].get('node_type'),
            }
            for node, score in ranked
        ]

    return {
        'betweenness_exact': betweenness_exact,
        'weighted_degree': top(weighted_degree),
        'pagerank': top(pagerank),
        'betweenness': top(betweenness),
    }

# TODO Q3:
# - create the required 2Wiki PMF/CCDF plots on log-log axes;
# - build the sensitivity and MuSiQue transfer tables;
# - visualize one complete multi-hop evidence path;
# - save figures under WORK_ROOT / 'figures'.
from collections import Counter

figures_dir = WORK_ROOT / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

def degree_pmf_ccdf(degrees):
    positive = [degree for degree in degrees if degree > 0]
    counts = Counter(positive)
    total = sum(counts.values())
    ks = sorted(counts)
    pmf = [counts[k] / total for k in ks]
    ccdf = [sum(c for kk, c in counts.items() if kk >= k) / total for k in ks]
    return ks, pmf, ccdf

stats_2wiki = {
    name: network_statistics(graph_sets['2wiki'][name])
    for name in ['native', 'unreconciled', 'exact_surface', 'conservative']
}
connectivity_table = pd.DataFrame([{'graph': name, **stats} for name, stats in stats_2wiki.items()])
display(connectivity_table)

degree_table = pd.DataFrame([
    {
        'graph': name,
        'degree_mean': stats['degree_mean'],
        'degree_max': stats['degree_max'],
        'average_clustering': stats['average_clustering'],
        'degree_assortativity': stats['degree_assortativity'],
    }
    for name, stats in stats_2wiki.items() if name != 'native'
])
display(degree_table)

centrality_tables = {}
for name in ['unreconciled', 'exact_surface', 'conservative']:
    audit = centrality_audit(graph_sets['2wiki'][name])
    centrality_tables[name] = pd.DataFrame({
        'weighted_degree': [row['name'] for row in audit['weighted_degree']],
        'pagerank': [row['name'] for row in audit['pagerank']],
        'betweenness': [row['name'] for row in audit['betweenness']],
    })
    display(centrality_tables[name])

fig, ax = plt.subplots()
for name in ['unreconciled', 'exact_surface', 'conservative']:
    degrees = [degree for _, degree in graph_sets['2wiki'][name].degree()]
    ks, pmf, _ = degree_pmf_ccdf(degrees)
    ax.plot(ks, pmf, marker='o', label=name)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('degree')
ax.set_ylabel('P(degree = k)')
ax.legend()
fig.savefig(figures_dir / '2wiki_degree_pmf.png', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots()
for name in ['unreconciled', 'exact_surface', 'conservative']:
    degrees = [degree for _, degree in graph_sets['2wiki'][name].degree()]
    ks, _, ccdf = degree_pmf_ccdf(degrees)
    ax.plot(ks, ccdf, marker='o', label=name)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('degree')
ax.set_ylabel('P(degree >= k)')
ax.legend()
fig.savefig(figures_dir / '2wiki_degree_ccdf.png', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots()
ax.bar(list(stats_2wiki.keys()), [stats['giant_component_fraction'] for stats in stats_2wiki.values()])
ax.set_ylabel('giant component fraction')
fig.savefig(figures_dir / '2wiki_giant_component_fraction.png', bbox_inches='tight')
plt.show()

gold_question = next(
    (
        row for row in packages['2wiki'].questions('public')
        if row.get('type') == 'bridge_comparison' and row.get('evidences')
    ),
    None,
)
if gold_question is None:
    print('No bridge_comparison question with evidences found for path plot.')
else:
    gold_path_graph = nx.DiGraph()
    for evidence in gold_question['evidences']:
        if not isinstance(evidence, (list, tuple)) or len(evidence) < 3:
            continue
        subject, relation, obj = evidence[:3]
        gold_path_graph.add_node(subject, node_type='entity')
        gold_path_graph.add_node(obj, node_type='entity')
        gold_path_graph.add_edge(subject, obj, label=relation)

    if gold_path_graph.number_of_nodes():
        pos = nx.spring_layout(gold_path_graph, seed=232)
        fig, ax = plt.subplots(figsize=(8, 6))
        nx.draw_networkx_nodes(gold_path_graph, pos, ax=ax)
        nx.draw_networkx_labels(gold_path_graph, pos, ax=ax, font_size=8)
        nx.draw_networkx_edges(gold_path_graph, pos, ax=ax)
        nx.draw_networkx_edge_labels(
            gold_path_graph, pos,
            edge_labels=nx.get_edge_attributes(gold_path_graph, 'label'), ax=ax, font_size=7,
        )
        ax.set_title(gold_question['question'])
        ax.axis('off')
        fig.savefig(figures_dir / '2wiki_gold_path.png', bbox_inches='tight')
        plt.show()
    else:
        print('Gold evidence was present but did not contain drawable triples.')

stats_musique = {
    name: network_statistics(graph_sets['musique'][name])
    for name in ['unreconciled', 'exact_surface', 'conservative']
}
musique_transfer_table = pd.DataFrame([
    {
        'graph': name,
        'components': stats['components'],
        'giant_component_fraction': stats['giant_component_fraction'],
        'isolated_nodes': stats['isolated_nodes'],
        'degree_mean': stats['degree_mean'],
        'degree_max': stats['degree_max'],
        'top5_weighted_degree': [
            row['name'] for row in centrality_audit(graph_sets['musique'][name], top_n=5)['weighted_degree']
        ],
    }
    for name, stats in stats_musique.items()
])
display(musique_transfer_table)

### Your written response for Question 3

Replace this cell with a **300–450 total-word explanation in your own words**.
Separate structure present in local GSWs from bridges introduced by reconciliation. Explain why degree alone cannot establish that a hub is semantically real, and do not claim a power law without a fitted comparison.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Stage A — Question 4: decomposition and dependency graphs (14 points)

This is the first neural stage. It loads only the 4B decomposer,
writes the raw response before parsing, and unloads the model when
finished. Implement validation before starting the 200-question
run. Invalid JSON, forward references, missing nodes, and cycles
must remain visible in the cache rather than being silently fixed.

In [ ]:
import re
PLACEHOLDER = re.compile(r'<ENTITY_Q(\d+)>')

def validate_decomposition(plan):
    # TODO Q4: return {'valid': bool, 'errors': [...], 'edges': [...]}.
    # Check schema, nonempty question text, reference bounds,
    # topological direction, and cycles.
    errors = []
    edges = []
    if not isinstance(plan, list) or not plan:
        return {'valid': False, 'errors': ['plan must be a nonempty list'], 'edges': []}

    for index, row in enumerate(plan, start=1):
        if not isinstance(row, dict) or 'question' not in row:
            errors.append(f'Q{index}: missing question field')
            continue
        question = row.get('question')
        if not isinstance(question, str) or not question.strip():
            errors.append(f'Q{index}: empty question text')
        requires_retrieval = row.get('requires_retrieval', True)
        if not isinstance(requires_retrieval, bool):
            errors.append(f'Q{index}: requires_retrieval must be boolean')
        for match in PLACEHOLDER.findall(str(question)):
            ref = int(match)
            if ref < 1 or ref >= index:
                errors.append(f'Q{index}: reference to Q{ref} is out of bounds or not earlier')
            else:
                edges.append((ref, index))

    return {'valid': len(errors) == 0, 'errors': errors, 'edges': edges}

def decomposition_metrics(predicted, reviewed):
    # TODO Q4: validity, subquestion-count exact match, dependency
    # edge precision/recall/F1, and retrieval/reasoning flag accuracy.
    predicted_validation = validate_decomposition(predicted)
    reviewed_validation = validate_decomposition(reviewed)

    predicted_edges = set(predicted_validation['edges'])
    reviewed_edges = set(reviewed_validation['edges'])
    true_positives = len(predicted_edges & reviewed_edges)
    precision = true_positives / len(predicted_edges) if predicted_edges else (1.0 if not reviewed_edges else 0.0)
    recall = true_positives / len(reviewed_edges) if reviewed_edges else (1.0 if not predicted_edges else 0.0)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    predicted_flags = [bool(row.get('requires_retrieval', True)) for row in predicted]
    reviewed_flags = [bool(row.get('requires_retrieval', True)) for row in reviewed]
    matched = min(len(predicted_flags), len(reviewed_flags))
    flag_matches = sum(predicted_flags[i] == reviewed_flags[i] for i in range(matched))
    flag_accuracy = flag_matches / len(reviewed_flags) if reviewed_flags else 1.0

    return {
        'predicted_valid': predicted_validation['valid'],
        'predicted_count': len(predicted),
        'reviewed_count': len(reviewed),
        'count_match': len(predicted) == len(reviewed),
        'edge_precision': precision,
        'edge_recall': recall,
        'edge_f1': f1,
        'retrieval_flag_accuracy': flag_accuracy,
    }

In [ ]:
# Stage A: restartable decomposition. Run only after the validator works.
if RUN_DECOMPOSITION_STAGE:
    from panini_course.qwen_models import QwenDecomposer

    model_cfg = json.loads((PACKAGE_ROOTS['2wiki'] / 'models/model_config.json').read_text())
    decomposer = QwenDecomposer(
        model_cfg['decomposer']['model'],
        PACKAGE_ROOTS['2wiki'] / model_cfg['decomposer']['prompt'],
        quantized=True,
    )
    try:
        for dataset, package in packages.items():
            output = WORK_ROOT / 'cache' / dataset / 'decompositions.jsonl'
            done = completed_ids(output)
            for question in selected_questions(package):
                qid = str(question['question_id'])
                if qid in done:
                    continue
                started = time.perf_counter()
                raw = decomposer.generate_raw(question['question'])
                record = {
                    'dataset': dataset,
                    'question_id': qid,
                    'question': question['question'],
                    'raw_response': raw,
                    'predicted_decomposition': None,
                    'decomposition_valid': False,
                    'validation_errors': [],
                    'seconds': time.perf_counter() - started,
                }
                try:
                    plan = decomposer.parse_response(raw)
                    validation = validate_decomposition(plan)
                    record.update({
                        'predicted_decomposition': plan,
                        'decomposition_valid': validation['valid'],
                        'validation_errors': validation['errors'],
                        'dependency_edges': validation['edges'],
                    })
                except Exception as error:
                    record['validation_errors'] = [repr(error)]
                append_jsonl(output, record)
                done.add(qid)
    finally:
        del decomposer
        release_gpu()
else:
    print('Stage A disabled. Set RUN_DECOMPOSITION_STAGE=True when ready.')

### Your written response for Question 4

Replace this cell with a **200–300-word explanation in your own words**.
Report the validation and dependency metrics, then classify the first irreversible error in representative failed plans. Explain the difference between a retrieval DAG and a memory graph.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Question 5 — sparse retrieval baselines (12 points)

Evaluate TF–IDF and BM25 for both direct QA retrieval and entity
retrieval followed by local GSW expansion. Use stable QA IDs for
relevance; do not evaluate by comparing array positions.

In [ ]:
from panini_course import BM25Index, TfidfIndex
from panini_course.metrics import recall_at_k, reciprocal_rank

def load_sparse_artifacts(root):
    return {
        'qa_tfidf': TfidfIndex.load(
            root/'indices/qa_tfidf.npz',
            root/'indices/qa_tfidf_vectorizer.joblib',
            root/'indices/qa_ids.json', source='qa_tfidf'),
        'qa_bm25': BM25Index.load(
            root/'indices/qa_bm25.joblib',
            root/'indices/qa_ids.json', source='qa_bm25'),
        'entity_tfidf': TfidfIndex.load(
            root/'indices/entity_tfidf.npz',
            root/'indices/entity_tfidf_vectorizer.joblib',
            root/'indices/entity_ids.json', source='entity_tfidf'),
        'entity_bm25': BM25Index.load(
            root/'indices/entity_bm25.joblib',
            root/'indices/entity_ids.json', source='entity_bm25'),
    }

sparse = {name: load_sparse_artifacts(root) for name, root in PACKAGE_ROOTS.items()}

def evaluate_sparse_retrieval(dataset, package, artifacts, k_values=(1, 5, 10, 15)):
    # TODO Q5: evaluate atomic gold retrieval tasks overall and by
    # question type/hop count. Include latency and failure examples.
    import time
    from panini_course.retrieval import SearchHit

    public_questions = package.questions('public')
    doc_labels = {}
    for row in public_questions:
        label = row.get('type') or row.get('hop_count')
        for doc_id in row.get('context_document_ids', []):
            doc_labels.setdefault(doc_id, set()).add(label)

    qa_pairs = package.qa_pairs()
    tasks = [
        {
            'query': row['question'],
            'gold': row['qa_uid'],
            'labels': sorted(doc_labels.get(row['document_id'], set())),
        }
        for row in qa_pairs
        if row['document_id'] in doc_labels
    ]

    qa_by_answer_local_id = {}
    qa_by_document_gsw = {}
    for row in qa_pairs:
        qa_by_document_gsw.setdefault((row['document_id'], row['gsw_file']), []).append(row)
        for local_id in row.get('answer_local_ids', []):
            qa_by_answer_local_id.setdefault(
                (row['document_id'], row['gsw_file'], local_id), []).append(row)

    def entity_expand(entity_hits, top_k):
        scored = {}
        for hit in entity_hits:
            parts = hit.item_id.split('::')
            if len(parts) != 3:
                continue
            document_id, gsw_file, local_id = parts
            for qa_row in qa_by_answer_local_id.get((document_id, gsw_file, local_id), []):
                scored[qa_row['qa_uid']] = max(scored.get(qa_row['qa_uid'], float('-inf')), hit.score)
            for qa_row in qa_by_document_gsw.get((document_id, gsw_file), []):
                scored.setdefault(qa_row['qa_uid'], hit.score * 0.5)
        ranked = sorted(scored.items(), key=lambda item: item[1], reverse=True)[:top_k]
        return [
            SearchHit(item_id=qa_uid, score=score, source='entity_bm25_expansion', rank=rank)
            for rank, (qa_uid, score) in enumerate(ranked, start=1)
        ]

    def run_method(search_fn):
        max_k = max(k_values)
        rows = []
        for task in tasks:
            started = time.perf_counter()
            hits = search_fn(task['query'], max_k)
            elapsed = time.perf_counter() - started
            ranked_ids = [hit.item_id for hit in hits]
            rank = ranked_ids.index(task['gold']) + 1 if task['gold'] in ranked_ids else None
            rows.append({
                'query': task['query'], 'gold': task['gold'],
                'labels': task['labels'], 'rank': rank, 'seconds': elapsed,
            })

        def summarize(subset):
            count = len(subset)
            if count == 0:
                summary = {f'recall@{k}': 0.0 for k in k_values}
                summary.update({'mrr': 0.0, 'mean_latency_seconds': 0.0, 'count': 0})
                return summary
            summary = {
                f'recall@{k}': sum(1 for row in subset if row['rank'] and row['rank'] <= k) / count
                for k in k_values
            }
            summary['mrr'] = sum(1.0 / row['rank'] if row['rank'] else 0.0 for row in subset) / count
            summary['mean_latency_seconds'] = sum(row['seconds'] for row in subset) / count
            summary['count'] = count
            return summary

        by_label = {}
        for label in sorted({label for row in rows for label in row['labels']}):
            by_label[label] = summarize([row for row in rows if label in row['labels']])

        failures = [row for row in rows if row['rank'] is None][:5]
        return {'overall': summarize(rows), 'by_label': by_label, 'failures': failures, 'rows': rows}

    tfidf_result = run_method(lambda query, top_k: artifacts['qa_tfidf'].search(query, top_k))
    bm25_result = run_method(lambda query, top_k: artifacts['qa_bm25'].search(query, top_k))
    entity_result = run_method(
        lambda query, top_k: entity_expand(artifacts['entity_bm25'].search(query, top_k), top_k))

    disagreements = [
        {
            'query': tfidf_row['query'], 'gold': tfidf_row['gold'],
            'tfidf_rank': tfidf_row['rank'], 'bm25_rank': bm25_row['rank'],
            'entity_expansion_rank': entity_row['rank'],
        }
        for tfidf_row, bm25_row, entity_row in zip(
            tfidf_result['rows'], bm25_result['rows'], entity_result['rows'])
        if (tfidf_row['rank'] is None) != (entity_row['rank'] is None)
    ][:5]

    return {
        'dataset': dataset,
        'tfidf': {
            'overall': tfidf_result['overall'], 'by_label': tfidf_result['by_label'],
            'failures': tfidf_result['failures'],
        },
        'bm25': {
            'overall': bm25_result['overall'], 'by_label': bm25_result['by_label'],
            'failures': bm25_result['failures'],
        },
        'entity_expansion': {
            'overall': entity_result['overall'], 'by_label': entity_result['by_label'],
            'failures': entity_result['failures'],
        },
        'disagreement_examples': disagreements,
    }

sparse_results = {
    dataset: evaluate_sparse_retrieval(dataset, packages[dataset], sparse[dataset])
    for dataset in packages
}

overall_rows = []
for dataset, result in sparse_results.items():
    for method in ['tfidf', 'bm25', 'entity_expansion']:
        overall_rows.append({'dataset': dataset, 'method': method, **result[method]['overall']})
overall_table = pd.DataFrame(overall_rows)
display(overall_table)

for dataset, result in sparse_results.items():
    for method in ['tfidf', 'bm25', 'entity_expansion']:
        print(dataset, method)
        display(pd.DataFrame(result[method]['by_label']).T)

for dataset, result in sparse_results.items():
    print(dataset, 'disagreement examples')
    display(pd.DataFrame(result['disagreement_examples']))

### Your written response for Question 5

Replace this cell with a **150–200-word explanation in your own words**.
Compare TF–IDF and BM25 using both aggregate metrics and concrete successes/failures. Explain which wording or token-frequency properties caused the difference.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Question 6 — dense, hybrid, and paper-style dual retrieval (16 points)

All corpus and required query embeddings are supplied. Load them by
stable ID and exact query text. A missing required query means your
deterministic plan or RICR path diverged; it is not permission to
generate a new embedding.

In [ ]:
from panini_course import (
    DenseIndex, DualRetriever, QueryEmbeddingStore,
    reciprocal_rank_fusion,
)

def load_dense_artifacts(root):
    query_store = QueryEmbeddingStore.load(
        root/'embeddings/query_embeddings.npy',
        root/'embeddings/query_ids.json',
        root/'embeddings/queries.jsonl')
    return {
        'queries': query_store,
        'qa_dense': DenseIndex.load(
            root/'indices/qa_qwen3_8b_ip.faiss',
            root/'indices/qa_ids.json', source='qa_dense'),
        'entity_dense': DenseIndex.load(
            root/'indices/entity_qwen3_8b_ip.faiss',
            root/'indices/entity_ids.json', source='entity_dense'),
    }

dense = {name: load_dense_artifacts(root) for name, root in PACKAGE_ROOTS.items()}

_dual_retrievers = {}

def _get_dual_retriever(dataset):
    if dataset not in _dual_retrievers:
        package = packages[dataset]
        _dual_retrievers[dataset] = DualRetriever(
            entity_index=sparse[dataset]['entity_bm25'],
            qa_index=dense[dataset]['qa_dense'],
            entity_rows=package.entities(),
            qa_rows=package.qa_pairs(),
        )
    return _dual_retrievers[dataset]

def retrieve_candidate_pool(dataset, query, pool_size=RETRIEVAL_POOL, backend='dual'):
    # TODO Q6:
    # - implement direct dense QA, RRF, and paper-style dual retrieval;
    # - dual retrieval is BM25 entity search + local GSW QA expansion
    #   unioned with direct dense QA retrieval;
    # - deduplicate by qa_uid and retain provenance/ranks;
    # - return QA records ready for Question 7 reranking.
    try:
        query_vector = dense[dataset]['queries'].get(query) if backend in ('dense', 'rrf', 'dual') else None
    except KeyError:
        print(f'Missing supplied query embedding for {dataset}: {query!r}')
        return []

    if backend == 'dense':
        hits = dense[dataset]['qa_dense'].search(query_vector, pool_size)
    elif backend == 'bm25':
        hits = sparse[dataset]['qa_bm25'].search(query, pool_size)
    elif backend == 'rrf':
        hits = reciprocal_rank_fusion(
            [
                sparse[dataset]['qa_tfidf'].search(query, pool_size),
                sparse[dataset]['qa_bm25'].search(query, pool_size),
                dense[dataset]['qa_dense'].search(query_vector, pool_size),
            ],
            rank_constant=RRF_CONSTANT,
            top_k=pool_size,
        )
    elif backend == 'dual':
        hits = _get_dual_retriever(dataset).search(
            query, query_vector=query_vector,
            entity_top_k=pool_size, qa_top_k=pool_size,
            fused_top_k=pool_size, rank_constant=RRF_CONSTANT,
        )
    else:
        raise ValueError(f'Unknown retrieval backend: {backend}')

    qa_metadata = packages[dataset].metadata_by_id('qa')
    pool = []
    seen = set()
    for rank, hit in enumerate(hits, start=1):
        if hit.item_id in seen:
            continue
        seen.add(hit.item_id)
        qa_row = qa_metadata.get(hit.item_id)
        if qa_row is None:
            continue
        pool.append({
            'qa_uid': hit.item_id,
            'question': qa_row['question'],
            'answer_names': tuple(qa_row.get('answer_names', ())),
            'answer_local_ids': tuple(qa_row.get('answer_local_ids', ())),
            'answer_role_states': tuple(qa_row.get('answer_role_states', ())),
            'document_id': qa_row['document_id'],
            'gsw_file': qa_row['gsw_file'],
            'score': hit.score,
            'source': hit.source,
            'rank': rank,
        })
        if len(pool) >= pool_size:
            break
    return pool

# TODO Q6: manually verify FAISS inner products for five supplied
# query vectors, then evaluate all retrieval variants consistently.
def verify_faiss_inner_products(dataset, sample_queries):
    qa_dense = dense[dataset]['qa_dense']
    query_store = dense[dataset]['queries']
    rows = []
    for query in sample_queries:
        vector = query_store.get(query)
        hits = qa_dense.search(vector, 1)
        if not hits:
            continue
        top_hit = hits[0]
        position = qa_dense.ids.index(top_hit.item_id)
        stored_vector = np.asarray(qa_dense.index.reconstruct(position), dtype=np.float32)
        normalized_query = vector / np.linalg.norm(vector)
        manual_score = float(np.dot(stored_vector, normalized_query))
        rows.append({
            'query': query,
            'qa_uid': top_hit.item_id,
            'faiss_score': top_hit.score,
            'manual_score': manual_score,
            'difference': abs(top_hit.score - manual_score),
        })
    return pd.DataFrame(rows)

for dataset in packages:
    sample_queries = [row['question'] for row in packages[dataset].questions('public')[:5]]
    display(verify_faiss_inner_products(dataset, sample_queries))

def evaluate_retrieval_backends(dataset, package, backends=('bm25', 'dense', 'rrf', 'dual'),
                                 pool_size=RETRIEVAL_POOL, k_values=(1, 5, 10, 15)):
    query_texts = {row['text'] for row in read_jsonl(PACKAGE_ROOTS[dataset] / 'embeddings' / 'queries.jsonl')}
    public_doc_ids = {
        doc_id
        for row in package.questions('public')
        for doc_id in row.get('context_document_ids', [])
    }
    tasks = [
        {'query': row['question'], 'gold': row['qa_uid']}
        for row in package.qa_pairs()
        if row['document_id'] in public_doc_ids and row['question'] in query_texts
    ]

    results = {}
    for backend in backends:
        rows = []
        for task in tasks:
            started = time.perf_counter()
            pool = retrieve_candidate_pool(dataset, task['query'], pool_size=pool_size, backend=backend)
            elapsed = time.perf_counter() - started
            ranked_ids = [row['qa_uid'] for row in pool]
            rank = ranked_ids.index(task['gold']) + 1 if task['gold'] in ranked_ids else None
            rows.append({'rank': rank, 'seconds': elapsed, 'pool_size': len(pool)})

        if not rows:
            results[backend] = {
                **{f'recall@{k}': 0.0 for k in k_values},
                'mrr': 0.0,
                'mean_latency_seconds': 0.0,
                'p95_latency_seconds': 0.0,
                'mean_candidate_count': 0.0,
                'task_count': 0,
            }
            continue
        latencies = sorted(row['seconds'] for row in rows)
        p95_index = min(len(latencies) - 1, int(0.95 * len(latencies)))
        results[backend] = {
            **{
                f'recall@{k}': sum(1 for row in rows if row['rank'] and row['rank'] <= k) / len(rows)
                for k in k_values
            },
            'mrr': sum(1.0 / row['rank'] if row['rank'] else 0.0 for row in rows) / len(rows),
            'mean_latency_seconds': sum(row['seconds'] for row in rows) / len(rows),
            'p95_latency_seconds': latencies[p95_index],
            'mean_candidate_count': sum(row['pool_size'] for row in rows) / len(rows),
            'task_count': len(rows),
        }
    return results

backend_results = {dataset: evaluate_retrieval_backends(dataset, packages[dataset]) for dataset in packages}
backend_rows = [
    {'dataset': dataset, 'backend': backend, **metrics}
    for dataset, results in backend_results.items()
    for backend, metrics in results.items()
]
backend_table = pd.DataFrame(backend_rows)
display(backend_table)

### Your written response for Question 6

Replace this cell with a **200–300-word explanation in your own words**.
Use the measured tables to explain where dense retrieval, RRF, and entity expansion help or hurt. Include candidate-set overlap and at least one error attributable to the retrieval pool rather than reranking.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Stage B — Question 7: reranking without exceeding Colab memory (8 points)

The reranker is the only neural model in this stage. Use the 8B
checkpoint in 4-bit mode when it fits; after an actual CUDA OOM,
clear memory and use the configured 4B fallback with batch size 1
and 256-token inputs. Cache rankings by exact instantiated query so
RICR replay and ablations do not rerun the same model call.

In [ ]:
from panini_course import Candidate

def format_qa_for_reranker(qa_row):
    # TODO Q7: format only the grounded QA record and its answer
    # role/state information. Do not pass source documents.
    answers = ', '.join(qa_row.get('answer_names', ()))
    role_states = '; '.join(qa_row.get('answer_role_states', ()))
    return f'Q: {qa_row["question"]} A: {answers} ({role_states})'

def rerank_and_convert(dataset, query, reranker, *, top_k=CANDIDATES_PER_HOP):
    # TODO Q7:
    # 1. retrieve a union pool with retrieve_candidate_pool;
    # 2. score its formatted QA records with QwenReranker;
    # 3. combine/calibrate retrieval and reranker scores as specified;
    # 4. return Candidate objects, one per QA, keeping all answers;
    # 5. namespace each local answer ID with document/GSW provenance.
    pool = retrieve_candidate_pool(dataset, query, pool_size=RETRIEVAL_POOL, backend='dual')
    if not pool:
        return []

    documents = [format_qa_for_reranker(row) for row in pool]
    probabilities = reranker.score(query, documents)

    candidates = []
    for row, probability in zip(pool, probabilities):
        reciprocal_rank = 1.0 / row['rank']
        combined = 0.5 * probability + 0.5 * reciprocal_rank
        candidates.append(
            Candidate(
                qa_uid=row['qa_uid'],
                answer_names=row['answer_names'],
                score=2.0 * combined - 1.0,
                question=row['question'],
                answer_ids=tuple(
                    f"{row['document_id']}::{row['gsw_file']}::{local_id}"
                    for local_id in row['answer_local_ids']
                ),
                answer_role_states=row['answer_role_states'],
                document_id=row['document_id'],
            )
        )
    candidates.sort(key=lambda candidate: candidate.score, reverse=True)
    return candidates[:top_k]

def choose_reranker_config(model_cfg):
    try:
        import torch
        total_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    except Exception:
        total_gib = 0.0
    use_8b = total_gib >= model_cfg['reranker']['use_8b_at_or_above_gib']
    return {
        'model': (model_cfg['reranker']['model'] if use_8b
                  else model_cfg['reranker']['free_colab_t4_fallback']),
        'batch_size': 1 if total_gib < 20 else 8,
        'max_length': 256 if total_gib < 20 else 2048,
        'total_gib': total_gib,
    }

### Your written response for Question 7

Replace this cell with a **150–200-word explanation in your own words**.
State whether reranking improved the candidate order and whether it changed complete-pool recall. Show a case where reranking demoted a strong exact match or promoted a semantically better candidate.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Question 8 — implement connected-DAG RICR (22 points)

Complete `panini_course/ricr.py`, not an alternative linear search
in this notebook. Your implementation must combine all parents at a
converging node, use harmonic-mean parent combinations and the 0.3
threshold/fallback, group intermediate answer entities, retain
final-hop QA alternatives, and gather evidence from all final
beams. Run the supplied tests before starting Stage B.

In [ ]:
# In Colab, edit PERSISTENT_RICR and PERSISTENT_TESTS through the
# file browser, then rerun from setup to sync ricr.py into the clone.
test_root = REPO_ROOT / 'tests'
test_result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(test_root)],
    check=False,
)
if STRICT_SMOKE_TESTS and test_result.returncode:
    raise RuntimeError(f'pytest failed with exit code {test_result.returncode}')

from panini_course import Candidate, run_panini_ricr

toy_plan = [
    {'question': 'Who directed Film A?', 'requires_retrieval': True},
    {'question': 'Who directed Film B?', 'requires_retrieval': True},
    {'question': 'Who was born later, <ENTITY_Q1> or <ENTITY_Q2>?',
     'requires_retrieval': True},
]
toy_table = {
    'Who directed Film A?': [Candidate('a1', ('Alice Director',), 0.8, answer_ids=('filmA::e1',))],
    'Who directed Film B?': [Candidate('b1', ('Bob Director',), 0.6, answer_ids=('filmB::e1',))],
    'Who was born later, Alice Director or Bob Director?': [Candidate('c1', ('Alice Director',), 0.9)],
}

def toy_retrieve_and_score(query, top_k):
    return toy_table.get(query, [])[:top_k]

try:
    toy_result = run_panini_ricr(
        toy_plan, toy_retrieve_and_score,
        original_question='Who was born later, Film A or Film B director?',
        beam_width=5, candidates_per_hop=15,
    )

    expected_score = (0.5 * (0.8 + 1.0) * 0.5 * (0.6 + 1.0) * 0.5 * (0.9 + 1.0)) ** (1 / 3)
    checks = [
        (
            toy_result.issued_queries == (
                'Who directed Film A?',
                'Who directed Film B?',
                'Who was born later, Alice Director or Bob Director?',
            ),
            'issued query sequence',
        ),
        ([step.qa_uid for step in toy_result.chains[0].steps] == ['a1', 'b1', 'c1'], 'chain QA IDs'),
        (abs(toy_result.chains[0].score - expected_score) < 1e-9, 'geometric chain score'),
        ({candidate.qa_uid for candidate in toy_result.evidence} == {'a1', 'b1', 'c1'}, 'deduplicated evidence'),
        (not toy_result.fallback, 'non-fallback execution'),
    ]
    failed = [name for ok, name in checks if not ok]
    if failed:
        message = 'RICR toy smoke check failed: ' + ', '.join(failed)
        if STRICT_SMOKE_TESTS:
            raise AssertionError(message)
        print(message)
    else:
        print('RICR toy smoke check passed.')
    print('hand-calculated score:', expected_score)
    print('RICR chain score:', toy_result.chains[0].score)
except (NotImplementedError, AttributeError, IndexError) as error:
    message = f'RICR toy smoke check skipped until panini_course/ricr.py is complete: {error!r}'
    if STRICT_SMOKE_TESTS:
        raise
    print(message)


In [ ]:
# Stage B: the completed RICR implementation drives dynamic queries.
if RUN_RERANK_AND_RICR_STAGE:
    from dataclasses import asdict
    from panini_course.qwen_models import QwenReranker

    model_cfg = json.loads((PACKAGE_ROOTS['2wiki'] / 'models/model_config.json').read_text())
    selected = choose_reranker_config(model_cfg)
    print('reranker selection:', selected)
    try:
        reranker = QwenReranker(
            selected['model'], quantized=True,
            max_length=selected['max_length'])
    except RuntimeError as error:
        if 'out of memory' not in str(error).casefold():
            raise
        release_gpu()
        selected.update({
            'model': model_cfg['reranker']['free_colab_t4_fallback'],
            'batch_size': 1,
            'max_length': 256,
        })
        reranker = QwenReranker(
            selected['model'], quantized=True, max_length=256)
    try:
        for dataset, package in packages.items():
            output = WORK_ROOT / 'cache' / dataset / 'ricr_traces.jsonl'
            done = completed_ids(output, configuration='default')
            plans = {row['question_id']: row for row in read_jsonl(
                WORK_ROOT / 'cache' / dataset / 'decompositions.jsonl')}
            for question in selected_questions(package):
                qid = str(question['question_id'])
                if qid in done:
                    continue
                plan_row = plans.get(qid)
                if not plan_row or not plan_row.get('decomposition_valid'):
                    append_jsonl(output, {
                        'dataset': dataset, 'configuration': 'default',
                        'question_id': qid, 'error': 'invalid decomposition'})
                    continue
                started = time.perf_counter()
                result = run_panini_ricr(
                    plan_row['predicted_decomposition'],
                    lambda query, k: rerank_and_convert(
                        dataset, query, reranker, top_k=k),
                    original_question=question['question'],
                    beam_width=BEAM_WIDTH,
                    candidates_per_hop=CANDIDATES_PER_HOP,
                    multi_dependency_threshold=MULTI_PARENT_THRESHOLD,
                )
                append_jsonl(output, {
                    'dataset': dataset,
                    'configuration': 'default',
                    'question_id': qid,
                    'question': question['question'],
                    'reranker_model': selected['model'],
                    'trace': asdict(result),
                    'retrieval_seconds': time.perf_counter() - started,
                    'error': None,
                })
                done.add(qid)
    finally:
        del reranker
        release_gpu()
else:
    print('Stage B disabled. Complete Questions 6–8 first.')

### Your written response for Question 8

Replace this cell with a **200–300-word explanation in your own words**.
Show your hand calculation and explain why independent linear branches are not equivalent to executing a converging retrieval DAG. Identify where beam diversity is gained or lost.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Question 9 — controlled RICR ablations (14 points)

Change one factor at a time on the fixed 20-question ablation slice.
Reuse cached candidate rankings whenever the instantiated query and
backend are unchanged. Do not compare a warm cache time with an
uncached model time as if they were equivalent.

In [ ]:
ablation_configs = [
    {'name': 'default', 'beam': 5, 'k': 15, 'backend': 'dual',
     'unique': True, 'score': 'geometric', 'parent_threshold': 0.3},
    {'name': 'beam_1', 'beam': 1},
    {'name': 'beam_3', 'beam': 3},
    {'name': 'k_5', 'k': 5},
    {'name': 'unique_off', 'unique': False},
    {'name': 'last_hop', 'score': 'last_hop'},
    {'name': 'parent_threshold_off', 'parent_threshold': 0.0},
    {'name': 'bm25', 'backend': 'bm25'},
    {'name': 'dense', 'backend': 'dense'},
    {'name': 'rrf', 'backend': 'rrf'},
]

def run_ablation(configuration, dataset, questions):
    # TODO Q9: merge each partial dictionary with the default,
    # execute exactly the same fixed questions, append traces, and
    # report chain recovery, answer metrics, evidence size, and a
    # valid cold or consistently cached latency measurement.
    default_config = ablation_configs[0]
    cfg = {**default_config, **configuration}

    def retrieve_fn(query, top_k):
        pool = retrieve_candidate_pool(dataset, query, pool_size=RETRIEVAL_POOL, backend=cfg['backend'])
        if not pool:
            return []
        documents = [format_qa_for_reranker(row) for row in pool]
        probabilities = reranker.score(query, documents)
        candidates = []
        for row, probability in zip(pool, probabilities):
            reciprocal_rank = 1.0 / row['rank']
            combined = 0.5 * probability + 0.5 * reciprocal_rank
            candidates.append(
                Candidate(
                    qa_uid=row['qa_uid'],
                    answer_names=row['answer_names'],
                    score=2.0 * combined - 1.0,
                    question=row['question'],
                    answer_ids=tuple(
                        f"{row['document_id']}::{row['gsw_file']}::{local_id}"
                        for local_id in row['answer_local_ids']
                    ),
                    answer_role_states=row['answer_role_states'],
                    document_id=row['document_id'],
                )
            )
        candidates.sort(key=lambda candidate: candidate.score, reverse=True)
        return candidates[:top_k]

    output = WORK_ROOT / 'cache' / dataset / 'ricr_traces.jsonl'
    done = completed_ids(output, configuration=cfg['name'])
    plans = {
        row['question_id']: row
        for row in read_jsonl(WORK_ROOT / 'cache' / dataset / 'decompositions.jsonl')
    }

    rows = []
    for question in questions:
        qid = str(question['question_id'])
        if qid in done:
            continue
        plan_row = plans.get(qid)
        if not plan_row or not plan_row.get('decomposition_valid'):
            record = {
                'dataset': dataset, 'configuration': cfg['name'],
                'question_id': qid, 'error': 'invalid decomposition',
            }
            append_jsonl(output, record)
            rows.append(record)
            continue

        started = time.perf_counter()
        result = run_panini_ricr(
            plan_row['predicted_decomposition'], retrieve_fn,
            original_question=question['question'],
            beam_width=cfg['beam'], candidates_per_hop=cfg['k'],
            multi_dependency_threshold=cfg['parent_threshold'],
            unique_intermediate_entities=cfg['unique'],
        )
        elapsed = time.perf_counter() - started

        best_chain = None
        if result.chains:
            rank_key = (
                (lambda chain: chain.score) if cfg['score'] == 'geometric'
                else (lambda chain: chain.last_hop_score)
            )
            best_chain = max(result.chains, key=rank_key)

        gold = [question['answer'], *question.get('answer_aliases', [])] if 'answer' in question else []
        predicted_answers = best_chain.current_answers if best_chain else ()
        gold_norm = {value.casefold().strip() for value in gold}
        answer_correct = bool(gold) and any(
            name.casefold().strip() in gold_norm for name in predicted_answers
        )

        record = {
            'dataset': dataset, 'configuration': cfg['name'], 'question_id': qid,
            'chain_recovered': bool(result.chains) and not result.fallback,
            'answer_correct': answer_correct,
            'evidence_size': len(result.evidence),
            'seconds': elapsed,
        }
        append_jsonl(output, record)
        rows.append(record)
        done.add(qid)

    valid_rows = [row for row in rows if 'error' not in row]
    count = len(valid_rows)
    return {
        'configuration': cfg['name'],
        'dataset': dataset,
        'questions': len(questions),
        'chain_recovery_rate': sum(row['chain_recovered'] for row in valid_rows) / count if count else 0.0,
        'answer_accuracy': sum(row['answer_correct'] for row in valid_rows) / count if count else 0.0,
        'mean_evidence_size': sum(row['evidence_size'] for row in valid_rows) / count if count else 0.0,
        'mean_seconds': sum(row['seconds'] for row in valid_rows) / count if count else 0.0,
    }

if RUN_ABLATIONS:
    require_full_run()
    # TODO Q9: use the first 20 public questions from each dataset.
    from panini_course.qwen_models import QwenReranker

    model_cfg = json.loads((PACKAGE_ROOTS['2wiki'] / 'models/model_config.json').read_text())
    selected = choose_reranker_config(model_cfg)
    try:
        reranker = QwenReranker(
            selected['model'], quantized=True, max_length=selected['max_length'])
    except RuntimeError as error:
        if 'out of memory' not in str(error).casefold():
            raise
        release_gpu()
        selected.update({
            'model': model_cfg['reranker']['free_colab_t4_fallback'],
            'batch_size': 1, 'max_length': 256,
        })
        reranker = QwenReranker(selected['model'], quantized=True, max_length=256)
    try:
        ablation_rows = []
        for dataset, package in packages.items():
            questions = package.questions('public')[:20]
            for partial_config in ablation_configs:
                summary = run_ablation(partial_config, dataset, questions)
                ablation_rows.append(summary)
                print(summary)
    finally:
        del reranker
        release_gpu()
    ablation_table = pd.DataFrame(ablation_rows)
    display(ablation_table)

### Your written response for Question 9

Replace this cell with a **250–350-word explanation in your own words**.
Use the measured table to recommend a configuration. Discuss accuracy, chain recovery, latency, and at least one interaction that a one-factor ablation cannot establish.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Stage C — Question 10: end-to-end 2Wiki evaluation (10 points)

Stage C reads saved RICR traces and loads only Qwen3-4B. The answer
model receives deduplicated QA evidence from every surviving final
beam, including role/state strings. It must not receive source
documents, graph neighbors, or gold labels. The supplied wrapper
already implements the required four-message one-shot PANINI prompt;
do not add an `N/A` instruction for these answerable splits.

In [ ]:
from panini_course.metrics import exact_match, token_f1

def format_evidence(candidate):
    # TODO Q10: produce one grounded line containing stored question,
    # all answer names, and answer role/state strings. Do not include
    # source passages or gold final answers.
    answers = ', '.join(candidate.get('answer_names', ()))
    role_states = '; '.join(candidate.get('answer_role_states', ()))
    return f"Q: {candidate['question']} A: {answers} ({role_states})"

if RUN_ANSWER_STAGE:
    from panini_course.qwen_models import QwenAnswerer

    model_cfg = json.loads((PACKAGE_ROOTS['2wiki'] / 'models/model_config.json').read_text())
    answerer = QwenAnswerer(model_cfg['answer_model']['model'], quantized=True)
    try:
        for dataset, package in packages.items():
            traces = {row['question_id']: row for row in read_jsonl(
                WORK_ROOT / 'cache' / dataset / 'ricr_traces.jsonl')
                if row.get('configuration') == 'default'}
            output = WORK_ROOT / 'cache' / dataset / 'answers.jsonl'
            done = completed_ids(output)
            for question in selected_questions(package):
                qid = str(question['question_id'])
                if qid in done or qid not in traces:
                    continue
                trace = traces[qid]
                if trace.get('error'):
                    continue
                evidence_rows = trace['trace']['evidence']
                evidence = [format_evidence(row) for row in evidence_rows]
                started = time.perf_counter()
                generated = answerer.answer_with_trace(question['question'], evidence)
                record = {
                    'dataset': dataset,
                    'question_id': qid,
                    'question': question['question'],
                    'predicted_answer': generated['answer'],
                    'raw_answer_response': generated['response'],
                    'answer_evidence': evidence,
                    'answer_seconds': time.perf_counter() - started,
                }
                if 'answer' in question:
                    gold = [question['answer'], *question.get('answer_aliases', [])]
                    record['exact_match'] = exact_match(generated['answer'], gold)
                    record['token_f1'] = token_f1(generated['answer'], gold)
                append_jsonl(output, record)
                done.add(qid)
    finally:
        del answerer
        release_gpu()
else:
    print('Stage C disabled. Complete and checkpoint Stage B first.')

In [ ]:
def score_default_run(dataset, package):
    # TODO Q10/Q11: join plans, traces, answers, and public labels by
    # question_id. Compute supporting-QA/document recall,
    # complete-chain recovery, EM/F1, evidence size, and latency;
    # summarize 2Wiki by type and MuSiQue by hop_count.
    from panini_course.metrics import normalize_answer, recall_at_k

    plans = {
        row['question_id']: row
        for row in read_jsonl(WORK_ROOT / 'cache' / dataset / 'decompositions.jsonl')
    }
    traces = {
        row['question_id']: row
        for row in read_jsonl(WORK_ROOT / 'cache' / dataset / 'ricr_traces.jsonl')
        if row.get('configuration') == 'default'
    }
    answers = {
        row['question_id']: row
        for row in read_jsonl(WORK_ROOT / 'cache' / dataset / 'answers.jsonl')
    }
    public_by_id = {str(row['question_id']): row for row in package.questions('public')}

    rows = []
    for qid, question in public_by_id.items():
        plan_row = plans.get(qid)
        trace_row = traces.get(qid)
        answer_row = answers.get(qid)

        row = {
            'question_id': qid,
            'type': question.get('type'),
            'hop_count': question.get('hop_count'),
            'decomposition_valid': bool(plan_row and plan_row.get('decomposition_valid')),
            'retrieval_error': bool(trace_row and trace_row.get('error')),
            'chain_recovered': False,
            'evidence_size': 0,
            'supporting_recall': None,
            'retrieval_seconds': None,
            'answer_seconds': None,
            'exact_match': None,
            'token_f1': None,
            'predicted_answer': None,
        }

        if trace_row and not trace_row.get('error'):
            trace = trace_row['trace']
            row['chain_recovered'] = bool(trace['chains']) and not trace['fallback']
            row['evidence_size'] = len(trace['evidence'])
            row['retrieval_seconds'] = trace_row.get('retrieval_seconds')

            evidence_doc_ids = {
                candidate['document_id'] for candidate in trace['evidence'] if candidate.get('document_id')
            }
            evidence_names = {
                normalize_answer(name)
                for candidate in trace['evidence']
                for name in candidate.get('answer_names', ())
            }
            if question.get('supporting_document_ids'):
                gold_ids = set(question['supporting_document_ids'])
                row['supporting_recall'] = (
                    recall_at_k(list(evidence_doc_ids), gold_ids, len(evidence_doc_ids))
                    if evidence_doc_ids else 0.0
                )
            elif question.get('supporting_facts'):
                gold_names = {normalize_answer(fact[0]) for fact in question['supporting_facts']}
                row['supporting_recall'] = (
                    recall_at_k(list(evidence_names), gold_names, len(evidence_names))
                    if evidence_names else 0.0
                )

        if answer_row:
            row['predicted_answer'] = answer_row.get('predicted_answer')
            row['answer_seconds'] = answer_row.get('answer_seconds')
            if 'answer' in question:
                gold = [question['answer'], *question.get('answer_aliases', [])]
                row['exact_match'] = exact_match(row['predicted_answer'] or '', gold)
                row['token_f1'] = token_f1(row['predicted_answer'] or '', gold)

        rows.append(row)

    return pd.DataFrame(rows)

# TODO Q10: display the overall and per-type 2Wiki tables, plus one
# successful and one failed trace with the first irreversible error.
scored_2wiki = score_default_run('2wiki', packages['2wiki'])

overall_2wiki = pd.DataFrame([{
    'questions': len(scored_2wiki),
    'decomposition_valid_rate': scored_2wiki['decomposition_valid'].mean(),
    'chain_recovery_rate': scored_2wiki['chain_recovered'].mean(),
    'mean_evidence_size': scored_2wiki['evidence_size'].mean(),
    'mean_supporting_recall': scored_2wiki['supporting_recall'].mean(),
    'exact_match': scored_2wiki['exact_match'].mean(),
    'token_f1': scored_2wiki['token_f1'].mean(),
    'mean_retrieval_seconds': scored_2wiki['retrieval_seconds'].mean(),
    'mean_answer_seconds': scored_2wiki['answer_seconds'].mean(),
}])
display(overall_2wiki)

by_type_2wiki = scored_2wiki.groupby('type').agg(
    questions=('question_id', 'count'),
    decomposition_valid_rate=('decomposition_valid', 'mean'),
    chain_recovery_rate=('chain_recovered', 'mean'),
    mean_evidence_size=('evidence_size', 'mean'),
    mean_supporting_recall=('supporting_recall', 'mean'),
    exact_match=('exact_match', 'mean'),
    token_f1=('token_f1', 'mean'),
)
display(by_type_2wiki)

successful = scored_2wiki[scored_2wiki['chain_recovered'] & (scored_2wiki['exact_match'] == 1.0)]
failed = scored_2wiki[(~scored_2wiki['chain_recovered']) | (scored_2wiki['retrieval_error'])]
if not successful.empty:
    print('successful trace:', successful.iloc[0].to_dict())
if not failed.empty:
    print('failed trace:', failed.iloc[0].to_dict())

### Your written response for Question 10

Replace this cell with a **250–350-word explanation in your own words**.
Interpret the 2Wiki result by failure stage: decomposition, candidate recall, later-hop substitution/pruning, or answer generation. A correct final answer does not by itself prove complete-chain recovery.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Question 11 — MuSiQue transfer and scaling (12 points)

Freeze all 2Wiki choices before inspecting MuSiQue results. Use the
same decomposition prompt, retrieval settings, beam rules,
reranker, answer prompt, and metrics. Compare 2-, 3-, and 4-hop
questions without tuning on MuSiQue labels.

In [ ]:
# TODO Q11: run score_default_run for MuSiQue, display results by
# hop_count, and plot both complete-chain recovery and answer F1 as
# chain length increases. Attribute errors using saved traces.
scored_musique = score_default_run('musique', packages['musique'])

musique_by_hop = scored_musique.groupby('hop_count').agg(
    questions=('question_id', 'count'),
    supporting_qa_recall=('supporting_recall', 'mean'),
    complete_chain_recovery=('chain_recovered', 'mean'),
    EM=('exact_match', 'mean'),
    F1=('token_f1', 'mean'),
).reset_index()
display(musique_by_hop)

fig, ax = plt.subplots()
ax.plot(musique_by_hop['hop_count'], musique_by_hop['complete_chain_recovery'], marker='o', label='complete chain recovery')
ax.plot(musique_by_hop['hop_count'], musique_by_hop['F1'], marker='o', label='answer F1')
ax.set_xlabel('hop count')
ax.set_ylabel('rate')
ax.legend()
fig.savefig(figures_dir / 'musique_hop_scaling.png', bbox_inches='tight')
plt.show()

error_rows = scored_musique[scored_musique['retrieval_error'] | (~scored_musique['decomposition_valid'])]
display(error_rows[['question_id', 'hop_count', 'decomposition_valid', 'retrieval_error', 'chain_recovered']].head(10))

### Your written response for Question 11

Replace this cell with a **250–350-word explanation in your own words**.
Explain why complete-chain recovery is conjunctive while answer F1 is not. Separate the effect of hop count from dataset wording, entity distribution, and GSW coverage.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Question 12 — reproducibility and final submission (12 points)

Materialize exactly four submission files: 80 labeled development
results and 20 label-free held-out predictions for each dataset.
Verify counts by stable question ID, not just line count. Record the
environment, frozen configuration, model IDs, resume instructions,
and which timings were cold versus cache replay.

In [ ]:
SUBMISSION_ROOT = WORK_ROOT / 'submission'
required_files = {
    SUBMISSION_ROOT/'results/2wiki_dev.jsonl': 80,
    SUBMISSION_ROOT/'predictions/2wiki_heldout.jsonl': 20,
    SUBMISSION_ROOT/'results/musique_dev.jsonl': 80,
    SUBMISSION_ROOT/'predictions/musique_heldout.jsonl': 20,
}

def materialize_submission(dataset, package):
    # TODO Q12: split cached answers by the package's public and
    # held-out ID sets. Development rows include metrics; held-out
    # rows contain only IDs, questions, predictions, and allowed
    # trace/runtime metadata. Never copy labels into held-out files.
    answers = {
        row['question_id']: row
        for row in read_jsonl(WORK_ROOT / 'cache' / dataset / 'answers.jsonl')
    }
    public_questions = package.questions('public')
    held_out_questions = package.questions('held_out')

    dev_path = SUBMISSION_ROOT / 'results' / f'{dataset}_dev.jsonl'
    dev_path.parent.mkdir(parents=True, exist_ok=True)
    with dev_path.open('w', encoding='utf-8') as handle:
        for question in public_questions:
            qid = str(question['question_id'])
            row = answers.get(qid)
            if row is None:
                continue
            record = {
                'question_id': qid,
                'question': row.get('question', question.get('question')),
                'predicted_answer': row.get('predicted_answer'),
                'exact_match': row.get('exact_match'),
                'token_f1': row.get('token_f1'),
                'answer_seconds': row.get('answer_seconds'),
            }
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')

    heldout_path = SUBMISSION_ROOT / 'predictions' / f'{dataset}_heldout.jsonl'
    heldout_path.parent.mkdir(parents=True, exist_ok=True)
    with heldout_path.open('w', encoding='utf-8') as handle:
        for question in held_out_questions:
            qid = str(question['question_id'])
            row = answers.get(qid)
            if row is None:
                continue
            record = {
                'question_id': qid,
                'question': row.get('question', question.get('question')),
                'predicted_answer': row.get('predicted_answer'),
                'answer_seconds': row.get('answer_seconds'),
            }
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')

def validate_submission_file(path, expected, allowed_heldout_fields=None):
    # TODO Q12: uniqueness, exact ID-set match, count, valid JSON,
    # finite metrics, and held-out field allowlist checks.
    import math

    rows = read_jsonl(path)
    ids = [row.get('question_id') for row in rows]
    assert len(ids) == len(set(ids)), f'{path}: duplicate question_id values'
    assert len(rows) == expected, f'{path}: expected {expected} rows, found {len(rows)}'

    dataset = 'musique' if 'musique' in path.name else '2wiki'
    package = packages[dataset]
    is_heldout = 'heldout' in path.name
    split = 'held_out' if is_heldout else 'public'
    expected_ids = {str(row['question_id']) for row in package.questions(split)}
    assert {str(value) for value in ids} == expected_ids, f'{path}: ID set does not match {split} split'

    for row in rows:
        for key, value in row.items():
            if isinstance(value, float):
                assert math.isfinite(value), f'{path}: non-finite value for {key}'

    if is_heldout:
        allowed = set(allowed_heldout_fields or {'question_id', 'question', 'predicted_answer', 'answer_seconds'})
        for row in rows:
            extra = set(row) - allowed
            assert not extra, f'{path}: disallowed held-out fields {extra}'

if QUESTION_LIMIT is None and all(path.exists() for path in required_files):
    for path, expected in required_files.items():
        validate_submission_file(path, expected)
    print('All required submission files validated.')
else:
    print('Final validation waits until QUESTION_LIMIT=None and all files exist.')

In [ ]:
# TODO Q12: write environment.txt and RUNME.md. Include at least:
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'question_limit': QUESTION_LIMIT,
    'beam_width': BEAM_WIDTH,
    'candidates_per_hop': CANDIDATES_PER_HOP,
    'retrieval_pool': RETRIEVAL_POOL,
    'rrf_constant': RRF_CONSTANT,
    'multi_parent_threshold': MULTI_PARENT_THRESHOLD,
    'package_manifests': {
        name: package.manifest for name, package in packages.items()
    },
}
(WORK_ROOT/'environment.json').write_text(
    json.dumps(environment, indent=2, ensure_ascii=False), encoding='utf-8')
(WORK_ROOT/'environment.txt').write_text(
    json.dumps(environment, indent=2, ensure_ascii=False), encoding='utf-8')
print('environment:', WORK_ROOT/'environment.json')
print('environment text:', WORK_ROOT/'environment.txt')

runme_lines = [
    '# RUNME',
    '',
    '## Commands',
    '1. `pip install -q -r requirements-colab.txt`',
    '2. Run all cells top to bottom with `QUESTION_LIMIT = 2` to smoke-test.',
    '3. Set `QUESTION_LIMIT = None`, enable exactly one of '
    '`RUN_DECOMPOSITION_STAGE` / `RUN_RERANK_AND_RICR_STAGE` / '
    '`RUN_ANSWER_STAGE` / `RUN_ABLATIONS` at a time, and rerun from the '
    'setup cell after each stage.',
    '4. Freeze the 2Wiki configuration before running the MuSiQue package.',
    '',
    '## Seeds and frozen configuration',
    f'- BEAM_WIDTH={BEAM_WIDTH}, CANDIDATES_PER_HOP={CANDIDATES_PER_HOP}, '
    f'RETRIEVAL_POOL={RETRIEVAL_POOL}, RRF_CONSTANT={RRF_CONSTANT}, '
    f'MULTI_PARENT_THRESHOLD={MULTI_PARENT_THRESHOLD}',
    '- Reconciliation audit sampling and centrality approximations use seed 232.',
    '',
    '## Expected runtime (free-tier T4)',
    '- Stage A (decomposer): roughly seconds per question.',
    '- Stage B (reranker + RICR): roughly seconds per question per configuration.',
    '- Stage C (answerer): roughly seconds per question.',
    '',
    '## Restart points',
    f'- `{WORK_ROOT / "cache"}/<dataset>/decompositions.jsonl`',
    f'- `{WORK_ROOT / "cache"}/<dataset>/ricr_traces.jsonl` (keyed by `configuration`)',
    f'- `{WORK_ROOT / "cache"}/<dataset>/answers.jsonl`',
    'Each file resumes by stable `question_id` and is safe to rerun after a disconnect.',
    '',
    '## Commit',
]
try:
    commit_hash = subprocess.run(
        ['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True,
        capture_output=True, text=True,
    ).stdout.strip()
except Exception:
    commit_hash = 'unknown'
runme_lines.append(f'- {commit_hash}')

(WORK_ROOT / 'RUNME.md').write_text('\n'.join(runme_lines), encoding='utf-8')
print('runme:', WORK_ROOT / 'RUNME.md')

### Your written response for Question 12

Replace this cell with a **300–450-word explanation in your own words**.
Tell the complete system story from input question to final answer, identify the design’s main protection and main failure mode, and explain how another student can resume and reproduce your run.

> **Write your response here.** Do not leave a list of numbers without
> interpreting what they mean and what could have caused them.

## Final pre-submission checklist

- [ ] `QUESTION_LIMIT` was set to `None` for final runs.
- [ ] All supplied tests pass, plus your reconciliation and failure-case tests.
- [ ] Every expensive stage resumes from JSONL by stable question ID.
- [ ] Only one Qwen model was resident at a time.
- [ ] No embeddings were generated.
- [ ] The 2Wiki configuration was frozen before MuSiQue evaluation.
- [ ] All twelve written-response cells were replaced with your own analysis.
- [ ] Held-out outputs contain no answer, alias, or supporting-evidence labels.
- [ ] Four required JSONL files, the executed notebook, figures, tests,
  `environment.json`, and `RUNME.md` are included.